# Lab 05.1 — Spark e database relazionali (JDBC / MySQL)

## Introduzione
Spark può leggere e scrivere qualunque database dotato di driver **JDBC**.
Usi tipici: importare tabelle in un data lake, unire dati del DB con file su HDFS/Parquet, scrivere risultati aggregati per applicazioni e report.

Il container `mysql` contiene il database `shop` con le tabelle `customers` (500 righe) e `orders` (20 000 righe), caricate da `initdb/01-shop.sql`.
Il driver JDBC viene scaricato da Maven al primo avvio con `spark.jars.packages` (conservato nel volume `ivy-cache`).

### Obiettivi
1. Inizializzare una `SparkSession` con il driver JDBC di MySQL (Step 1).
2. Leggere tabelle del database come DataFrame (Step 2).
3. Sfruttare il *pushdown* di filtri, proiezioni e query verso il database (Step 3).
4. Parallelizzare la lettura con le opzioni di partizionamento (Step 4).
5. Unire i dati del DB con altre sorgenti e analizzarli in Spark (Step 5).
6. Scrivere i risultati su nuove tabelle MySQL (Step 6).
7. Chiudere la sessione (Step 7).

## Step 1: Inizializzare la SparkSession con il driver JDBC

La `SparkSession` è il punto di ingresso per DataFrame e SQL. Il master è letto dalla variabile d'ambiente `SPARK_MASTER` (nel lab è `local[*]`: i thread girano dentro il driver Jupyter).
Con `spark.jars.packages` Spark scarica da Maven il driver `mysql-connector-j` e lo mette in `spark.jars.ivy` (cartella montata sul volume `ivy-cache`, così il download avviene solo al primo avvio).
Definiamo anche l'URL JDBC e le credenziali, riusate in tutte le celle successive.

In [ ]:
import os, time
from pyspark.sql import SparkSession, functions as F

# Sessione Spark: il pacchetto Maven con il driver JDBC viene scaricato al primo avvio
spark = (SparkSession.builder
         .appName("Lab05-MySQL")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.jars.packages", "com.mysql:mysql-connector-j:9.5.0")
         .config("spark.jars.ivy", "/home/jovyan/.ivy2")   # cache persistente dei pacchetti
         .getOrCreate())

# URL JDBC (host "mysql" = nome del servizio Compose) e credenziali dell'utente di esempio
URL = os.environ.get("MYSQL_URL", "jdbc:mysql://mysql:3306/shop")
PROPS = {"user": "spark", "password": "spark", "driver": "com.mysql.cj.jdbc.Driver"}

## Step 2: Leggere una tabella

`spark.read.jdbc(url, tabella, properties)` crea un DataFrame a partire da una tabella MySQL.
Lo schema viene ricavato dai metadati del database (tipi SQL → tipi Spark). Senza opzioni di partizionamento Spark apre **una sola connessione**: il DataFrame ha una sola partizione e la lettura è seriale.

In [ ]:
# Lettura delle due tabelle del database "shop"
orders = spark.read.jdbc(URL, "orders", properties=PROPS)
customers = spark.read.jdbc(URL, "customers", properties=PROPS)

# Schema dedotto dai metadati del DB e anteprima dei dati
orders.printSchema()
orders.show(5)
print("Partizioni:", orders.rdd.getNumPartitions(), "(una sola connessione: lettura seriale)")

### Punti chiave
- Una tabella JDBC diventa un DataFrame con schema dedotto dal database (tipi SQL → tipi Spark).
- Senza opzioni di partizionamento la lettura usa **una sola connessione** e una sola partizione.

## Step 3: Pushdown verso il database

Spark traduce filtri e proiezioni in SQL eseguito **da MySQL**: in Spark arrivano solo le righe e le colonne necessarie.
Nel piano di esecuzione (`explain()`) il pushdown si riconosce dalla voce `PushedFilters`.

### Step 3a: Pushdown di filtri e proiezioni

Applichiamo `where` e `select` come su un normale DataFrame: il filtro su `status` e `amount` viene eseguito dal database, quindi `count()` lavora solo sulle righe già filtrate.

In [ ]:
# Filtro e proiezione: verranno tradotti in una query SQL eseguita da MySQL
pagati = orders.where("status = 'PAID' AND amount > 200").select("order_id", "amount")
pagati.explain()      # PushedFilters: [IsNotNull(status), EqualTo(status,PAID), GreaterThan(amount,200)]
print("Righe:", pagati.count())

### Step 3b: Delegare al database un'intera query

Con l'opzione `query` si può far eseguire a MySQL una query SQL completa (qui un join con aggregazione): Spark riceve soltanto il risultato, già ridotto a poche righe. È utile quando il database è più efficiente di Spark per l'operazione (indici, join locali).

In [ ]:
# Si può anche delegare al DB un'intera query con l'opzione "query"
per_citta = (spark.read.format("jdbc")
             .option("url", URL).options(**PROPS)
             .option("query", """
                 SELECT c.city, COUNT(*) AS n_ordini, SUM(o.amount) AS totale
                 FROM orders o JOIN customers c ON o.customer_id = c.customer_id
                 GROUP BY c.city""")
             .load())
# L'ordinamento finale è invece eseguito da Spark
per_citta.orderBy(F.desc("totale")).show()

### Punti chiave
- Filtri e proiezioni dei DataFrame diventano SQL eseguito dal database (`PushedFilters` nel piano).
- L'opzione `query` delega a MySQL un'intera query: Spark scarica solo il risultato.

## Step 4: Lettura parallela

Indicando una colonna numerica e il suo intervallo (`column`, `lowerBound`, `upperBound`) Spark divide la tabella in `numPartitions` range ed esegue **una query per partizione in parallelo**:
`WHERE order_id >= 1 AND order_id < 5001`, `WHERE order_id >= 5001 AND ...`, ...
Gli estremi servono solo a calcolare i range: le righe fuori intervallo finiscono comunque nella prima o nell'ultima partizione.

In [ ]:
# 4 partizioni sul campo order_id (1..20000): 4 query in parallelo, una per range
orders_par = spark.read.jdbc(URL, "orders", properties=PROPS,
                             column="order_id", lowerBound=1, upperBound=20001, numPartitions=4)
print("Partizioni:", orders_par.rdd.getNumPartitions())
# Numero di righe in ciascuna partizione (circa 5000 ciascuna)
print("Righe per partizione:", orders_par.rdd.glom().map(len).collect())

### Punti chiave
- `column` + `lowerBound` + `upperBound` + `numPartitions` generano una query per partizione, eseguite in parallelo.
- La colonna di partizionamento deve essere numerica (o data/timestamp) e distribuita in modo uniforme.

## Step 5: Analisi in Spark e unione con altri dati

Una volta caricati, i dati del DB sono normali DataFrame: si possono unire a dati provenienti da altre sorgenti (qui un piccolo DataFrame con gli obiettivi di vendita per città).
Calcoliamo il venduto per città (escludendo gli ordini annullati) e la percentuale di obiettivo raggiunta.

In [ ]:
# DataFrame locale con gli obiettivi di vendita per città (seconda sorgente dati)
obiettivi = spark.createDataFrame(
    [("Roma", 300_000), ("Milano", 450_000), ("Napoli", 250_000),
     ("Torino", 420_000), ("Bologna", 260_000), ("Firenze", 500_000)],
    "city STRING, obiettivo BIGINT")

# Join ordini-clienti (da MySQL), aggregazione per città, join con gli obiettivi
risultati = (orders_par.where("status != 'CANCELLED'")
             .join(customers, "customer_id")
             .groupBy("city").agg(F.sum("amount").alias("venduto"))
             .join(obiettivi, "city")
             .withColumn("raggiunto_%", F.round(F.col("venduto") / F.col("obiettivo") * 100, 1))
             .orderBy(F.desc("raggiunto_%")))
risultati.show()

### Punti chiave
- I DataFrame provenienti da JDBC si combinano liberamente con qualunque altra sorgente (file, DataFrame locali, altri database).

## Step 6: Scrivere su MySQL

`write.jdbc` salva un DataFrame in una tabella MySQL. `mode("overwrite")` ricrea la tabella; `append` aggiunge righe.
Ogni partizione scrive in parallelo con la propria connessione, con inserimenti a blocchi di `batchsize` righe.

### Step 6a: Scrivere i risultati e rileggerli

Salviamo il DataFrame `risultati` nella tabella `risultati_citta` e lo rileggiamo dal database per verificare la scrittura.

In [ ]:
# Scrittura: la tabella viene ricreata (overwrite), inserimenti a blocchi di 1000 righe
(risultati.write.mode("overwrite")
          .option("batchsize", 1000)
          .jdbc(URL, "risultati_citta", properties=PROPS))

# Rilettura dal database per verifica
spark.read.jdbc(URL, "risultati_citta", properties=PROPS).show()

### Step 6b: Statistiche mensili in una nuova tabella

Aggreghiamo gli ordini per mese e stato e scriviamo il risultato nella tabella `statistiche_mensili`, mostrando quante righe sono state scritte.

In [ ]:
# Statistiche mensili scritte in una nuova tabella
mensili = (orders_par.groupBy(F.date_format("order_date", "yyyy-MM").alias("mese"), "status")
           .agg(F.count("*").alias("n"), F.round(F.sum("amount"), 2).alias("totale")))
mensili.write.mode("overwrite").jdbc(URL, "statistiche_mensili", properties=PROPS)
print(spark.read.jdbc(URL, "statistiche_mensili", properties=PROPS).count(), "righe scritte")

### Punti chiave
- `mode("overwrite")` ricrea la tabella, `append` aggiunge righe; ogni partizione scrive con la propria connessione.
- `batchsize` controlla quante righe vengono inviate per ogni INSERT a blocchi.

## Step 7: Chiudere la SparkSession

Termina l'applicazione Spark e libera le risorse (e le connessioni JDBC) del driver.

In [ ]:
# Chiude la sessione e libera le risorse
spark.stop()

## Riepilogo

| Concetto | API o comando | A cosa serve |
|---|---|---|
| Driver JDBC | `spark.jars.packages`, `spark.jars.ivy` | scaricare da Maven il driver MySQL e conservarlo nella cache |
| Lettura di una tabella | `spark.read.jdbc(url, tabella, properties)` | caricare una tabella come DataFrame (una sola partizione) |
| Pushdown | `where`, `select`, `explain()` / `PushedFilters` | far eseguire filtri e proiezioni al database |
| Query delegata | `.format("jdbc").option("query", ...)` | far eseguire a MySQL un'intera query SQL |
| Lettura parallela | `column`, `lowerBound`, `upperBound`, `numPartitions` | una query per partizione, in parallelo |
| Unione con altri dati | `join`, `groupBy`, `agg` | combinare il DB con altre sorgenti |
| Scrittura | `write.mode("overwrite"/"append").jdbc(...)`, `batchsize` | salvare risultati su tabelle MySQL |
| Chiusura | `spark.stop()` | terminare la sessione |